# Microsoft Agent Framework — Azure OpenAI (Responses API)

ในตัวอย่างโค้ดนี้ คุณจะใช้ **Microsoft Agent Framework (MAF)** เพื่อสร้างเอเจนต์ง่ายๆ โดยใช้ **Azure OpenAI** ผ่าน **Responses API**

> **หมายเหตุการย้าย:** ตัวอย่างนี้เคยใช้ Semantic Kernel กับ GitHub Models มาก่อน และได้ย้ายมาใช้ Microsoft Agent Framework แทน โดย GitHub Models (เลิกใช้ และจะเลิกให้บริการในเดือนกรกฎาคม 2026) ได้ถูกแทนที่ด้วย Azure OpenAI ซึ่งรองรับ Responses API `OpenAIChatClient` ใน MAF จะเชื่อมต่อกับจุดปลายทางที่เสถียรของ Azure OpenAI ที่ `/openai/v1/` และใช้ Responses API เป็นค่าเริ่มต้น

จุดประสงค์ของตัวอย่างนี้คือเพื่อแสดงขั้นตอนที่จะนำไปใช้ในตัวอย่างโค้ดเพิ่มเติมในอนาคตเมื่อทำงานกับรูปแบบเอเจนต์ต่างๆ


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## นำเข้าแพ็กเกจ Python ที่จำเป็น


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## การกำหนดเครื่องมือ

ใน Microsoft Agent Framework, **เครื่องมือ** คือฟังก์ชัน Python ธรรมดาที่ติดตั้งด้วย `@tool` ที่เอเจนต์สามารถเรียกใช้ได้ ด้านล่างเราจะกำหนดเครื่องมือที่ส่งคืนจุดหมายปลายทางการพักผ่อนแบบสุ่มและหลีกเลี่ยงการซ้ำกับจุดหมายปลายทางก่อนหน้า


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## การสร้าง Agent

ที่นี่ เราจะสร้าง Agent ที่ชื่อ `TravelAgent`

ในตัวอย่างนี้ เราใช้คำสั่งพื้นฐานมาก ๆ คุณสามารถแก้ไขคำสั่งเหล่านี้เพื่อดูว่าพฤติกรรมของ agent เปลี่ยนแปลงอย่างไร


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## การรันเอเจนต์

ตอนนี้เราสามารถรันเอเจนต์ได้แล้ว เราสร้าง `AgentSession` เพื่อให้เอเจนต์จำการสนทนาในแต่ละรอบได้ จากนั้นส่ง `user_inputs` สองข้อความ ข้อความแรกขอทริป; ข้อความที่สองบอกว่าผู้ใช้ไม่ชอบคำแนะนำและขออีกอัน — เอเจนต์ใช้ประวัติการสนทนาในเซสชันร่วมกับเครื่องมือ `get_random_destination` เพื่อตอบกลับ

คุณสามารถแก้ไขข้อความเหล่านี้เพื่อสังเกตว่าเอเจนต์ตอบสนองแตกต่างกันอย่างไร การตอบกลับจะถูก **สตรีม** ทีละโทเค็น


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ปฏิเสธความรับผิดชอบ**:
เอกสารนี้ได้รับการแปลโดยใช้บริการแปลภาษา AI [Co-op Translator](https://github.com/Azure/co-op-translator) ขณะที่เราพยายามให้ความถูกต้อง โปรดทราบว่าการแปลโดยอัตโนมัติอาจมีข้อผิดพลาดหรือความไม่ถูกต้อง เอกสารต้นฉบับในภาษาต้นทางควรถูกพิจารณาเป็นแหล่งข้อมูลที่เชื่อถือได้ สำหรับข้อมูลที่สำคัญ แนะนำให้ใช้การแปลโดยมนุษย์มืออาชีพ เราไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความที่ผิดพลาดที่เกิดขึ้นจากการใช้การแปลนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
